# Phase 2: Data Cleaning & Domain Standardization
## End-to-End Cricket Analytics & Performance Intelligence

### 1. Objective
This notebook demonstrates the cricket domain standardization pipeline. In professional sports analytics, generic data cleaning without cricket domain knowledge corrupts key performance indicators (KPIs).

We address:
1. **Franchise Rebrands & Entity Resolution**: Unifying historical name changes (e.g. *Delhi Daredevils* $\rightarrow$ *Delhi Capitals*, *Kings XI Punjab* $\rightarrow$ *Punjab Kings*, *Royal Challengers Bangalore* $\rightarrow$ *Royal Challengers Bengaluru*).
2. **Venue & City Imputation**: Deduplicating naming conventions (e.g. *Feroz Shah Kotla* vs *Arun Jaitley Stadium*) and imputing missing host cities.
3. **Delivery Legality & Runs Allocation**: Isolating true balls faced by batters (excluding wides) and true balls bowled by bowlers (excluding wides and no-balls).
4. **True Bowler Runs Conceded**: Byes and leg-byes are fielding/wicket-keeping errors and are not charged to the bowler's economy.
5. **Dismissal Classification**: Separating bowler wickets (*bowled, caught, lbw, stumped, hit wicket*) from non-bowler dismissals (*run out, retired hurt*) for proper bowling average and strike rate calculations.
6. **Match Phase Segmentation**: Powerplay (Overs 1–6), Middle (Overs 7–15), Death (Overs 16–20).

In [1]:
import sys
import os
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..')))
from src.cleaner import run_cleaning_pipeline

# Check if cleaned data already exists from main pipeline run
matches_clean_path = '../data/processed/matches_clean.csv'
deliv_clean_path = '../data/processed/deliveries_clean.csv'

FORCE_RECLEAN = False  # Set to True if you want to re-run from scratch

if FORCE_RECLEAN or not (os.path.exists(matches_clean_path) and os.path.exists(deliv_clean_path)):
    print("Running full cleaning pipeline...")
    run_cleaning_pipeline()
else:
    print("[OK] Cleaned datasets already exist in data/processed/! Loading directly for fast inspection...")

[OK] Cleaned datasets already exist in data/processed/! Loading directly for fast inspection...


### 2. Loading Cleaned Datasets
We load the cleaned `matches_clean.csv` and a sample of `deliveries_clean.csv` for high-speed inspection.

In [2]:
df_matches = pd.read_csv('../data/processed/matches_clean.csv')
df_deliv = pd.read_csv('../data/processed/deliveries_clean.csv', nrows=10000)

print(f"Matches dataset:    {df_matches.shape[0]:,} matches | {df_matches.shape[1]} columns")
print(f"Deliveries sample:  {df_deliv.shape[0]:,} deliveries | {df_deliv.shape[1]} columns")
display(df_matches[['season', 'date', 'team1', 'team2', 'winner', 'win_type', 'margin', 'margin_unit']].head(5))

Matches dataset:    1,243 matches | 19 columns
Deliveries sample:  10,000 deliveries | 33 columns


,season,date,team1,team2,winner,win_type,margin,margin_unit
0,2008,2008-04-18,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders,Defending,140,Runs
1,2008,2008-04-19,Punjab Kings,Chennai Super Kings,Chennai Super Kings,Defending,33,Runs
2,2008,2008-04-19,Delhi Capitals,Rajasthan Royals,Delhi Capitals,Chasing,9,Wickets
3,2008,2008-04-20,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru,Chasing,5,Wickets
4,2008,2008-04-20,Kolkata Knight Riders,Deccan Chargers,Kolkata Knight Riders,Chasing,5,Wickets


### 3. Demonstration of Domain Corrections

#### A. Franchise Name Harmonization
Notice how historical franchise rebrands have been unified so that franchise analytics remain continuous across all 17 seasons.

In [3]:
print("Standardized Franchise List (15 Unique Teams):")
for t in sorted(df_matches['team1'].unique()):
    print(f" - {t}")

Standardized Franchise List (15 Unique Teams):
 - Chennai Super Kings
 - Deccan Chargers
 - Delhi Capitals
 - Gujarat Lions
 - Gujarat Titans
 - Kochi Tuskers Kerala
 - Kolkata Knight Riders
 - Lucknow Super Giants
 - Mumbai Indians
 - Pune Warriors
 - Punjab Kings
 - Rajasthan Royals
 - Rising Pune Supergiant
 - Royal Challengers Bengaluru
 - Sunrisers Hyderabad


#### B. Delivery Legality & Runs Allocation
In T20 cricket:
- `is_ball_faced`: Excludes wides (a wide does not consume a ball faced by the batsman).
- `is_legal_ball`: Excludes both wides and no-balls (does not count toward the 6-ball over quota).
- `bowler_runs`: Excludes byes and legbyes (fielding lapses, not charged to bowler's economy).

In [4]:
cols = ['striker', 'bowler', 'runs_off_bat', 'extras', 'total_runs', 'bowler_runs', 'is_ball_faced', 'is_legal_ball', 'is_dot_ball']
display(df_deliv[cols].head(5))

,striker,bowler,runs_off_bat,extras,total_runs,bowler_runs,is_ball_faced,is_legal_ball,is_dot_ball
0,DA Warner,TS Mills,0,0,0,0,1,1,1
1,DA Warner,TS Mills,0,0,0,0,1,1,1
2,DA Warner,TS Mills,4,0,4,4,1,1,0
3,DA Warner,TS Mills,0,0,0,0,1,1,1
4,DA Warner,TS Mills,0,2,2,2,0,0,0


#### C. Bowler Wickets vs Fielding Dismissals
Only true bowler wickets (`bowled`, `caught`, `caught and bowled`, `lbw`, `stumped`, `hit wicket`) count toward bowling average and strike rate. Run outs are strictly excluded.

In [5]:
wicket_sample = df_deliv[df_deliv['is_wicket'] == 1][['match_id', 'striker', 'bowler', 'wicket_type', 'is_bowler_wicket', 'is_run_out']]
display(wicket_sample.head(6))

,match_id,striker,bowler,wicket_type,is_bowler_wicket,is_run_out
11,1082591,DA Warner,A Choudhary,caught,1,0
64,1082591,S Dhawan,STR Binny,caught,1,0
94,1082591,MC Henriques,YS Chahal,caught,1,0
116,1082591,Yuvraj Singh,TS Mills,bowled,1,0
160,1082591,Mandeep Singh,Rashid Khan,bowled,1,0
165,1082591,CH Gayle,DJ Hooda,caught,1,0


#### D. Match Phase Segmentation
Deliveries are partitioned into Powerplay (Overs 1–6), Middle Overs (Overs 7–15), and Death Overs (Overs 16–20).

In [6]:
phase_breakdown = df_deliv['match_phase'].value_counts()
display(phase_breakdown.to_frame(name='Delivery Count'))

,Delivery Count
match_phase,
Middle,4592
Powerplay,3174
Death,2221
Super Over,13


### 4. Summary & Verification Status
The data cleaning pipeline has satisfied all domain requirements. The output tables are ready for Star Schema modeling in Phase 3.